# Travel Reimbursement Approval Agent
**Shaswat Prakash · Candidate assignment · USD**

Evaluate the five supplied mock claims against category eligibility, receipt requirements, daily/nightly limits, submission dates, and approval authority. Each recommendation includes exact-cent calculations, stable reason codes, and citations to the assignment's policy.

## README — setup and execution

Open this notebook in JupyterLab or Google Colab with **Python 3.11 or later**, then choose **Run All**. All policy text, input claims, evaluation code, checks, and results are embedded. No repository imports, external input files, credentials, or API keys are required for this execution.

The first cell installs **Pydantic 2.13.5** only when the pinned dependency is absent. Initial installation requires internet access. A preconfigured environment avoids installation. No interactive prompts or manual intervention are required during Run All.

**Execution mode:** deterministic Python policy evaluation. The results shown here are calculated by policy functions; no model inference or MCP traffic is claimed by this run. The policy layer can be reused by a model-connected tool workflow without changing the monetary rules.

**Run order:** configuration → supplied policy/claims → typed contracts → policy functions → input verification → generated recommendations/evidence → automated checks → design notes → final JSON array.

The last code cell returns one object for each of the five supplied claims with exactly the nine required fields. All employee names and claims are mock data from the assignment.


In [1]:
import importlib.metadata
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Use a Python 3.11+ kernel before running this notebook.")
try:
    installed_pydantic = importlib.metadata.version("pydantic")
except importlib.metadata.PackageNotFoundError:
    installed_pydantic = None
if installed_pydantic != "2.13.5":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pydantic==2.13.5"])
print("Runtime ready: Python", sys.version.split()[0], "· Pydantic", importlib.metadata.version("pydantic"))


Runtime ready: Python 3.12.13 · Pydantic 2.13.5


## Appendix A — authoritative policy and limits

The twelve `POL-*` identifiers and rule text below are transcribed from **Appendix A, page 3** of the supplied *AI Developer Candidate Assignment — Travel Reimbursement Approval Agent*. Formatting is normalized; monetary limits, receipt rules, timeliness, and approval tiers are preserved.

Approval thresholds use the **reimbursable amount after deductions**, not the gross claim. The assignment treats fully compliant manager-tier claims as approvable. Every decision-driving finding cites its applicable rule; technical consistency checks do not invent a policy ID.

## Appendix B — supplied claims

`SAMPLE_CLAIMS` preserves the five supplied IDs, names, descriptions, dates, amounts, and receipt attachment flags from **Appendix B, page 4**. Structured metadata represents information stated in the descriptions: airfare class and category-specific day/night counts. These remain the only demonstration claims.


In [2]:
from __future__ import annotations

import asyncio
import copy
import hashlib
import html
import json
import os
import re
import time
from collections import defaultdict
from datetime import date, datetime, timezone
from decimal import Decimal, InvalidOperation
from enum import Enum
from typing import Any, Literal, Protocol

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

POLICIES = {
    "POL-CAT-01": "Eligible categories — Reimbursable when incurred for a documented business purpose: Airfare (economy class only — see POL-AIR-01); Lodging (hotel room charges); Meals (subject to per-diem limits — see POL-PD-01); Ground transport (taxi, rideshare, train, rental car, parking); Conference / registration fees.",
    "POL-CAT-02": "Ineligible items — Never reimbursable; rejected (deducted in full): Alcohol and minibar charges; Spa, gym, and personal entertainment; In-room movies, personal shopping, gifts; Traffic fines, penalties, and late fees; Any personal (non-business) expense.",
    "POL-PD-01": "Meals — Maximum $75 per day. Amounts above the daily cap are deducted; the rest is reimbursed.",
    "POL-PD-02": "Lodging — Maximum $200 per night. Amounts above the nightly cap are deducted; the rest is reimbursed.",
    "POL-PD-03": "Ground transport — Maximum $50 per day. Amounts above the cap are deducted.",
    "POL-AIR-01": "Airfare class — Only economy class airfare is reimbursable. Business/first-class fares are a policy exception and must be routed to Manual Review (not auto-deducted, because a pre-approval may exist).",
    "POL-RCT-01": "Receipt required above $25 — Any single line item greater than $25 requires an attached, itemized receipt. Airfare and lodging always require a receipt regardless of amount.",
    "POL-RCT-02": "Missing receipt handling — If a receipt is missing for an item that requires one, the item is not silently rejected — the claim is routed to Manual Review so the reviewer can request the receipt.",
    "POL-APR-01": "Auto-approve tier — Total ≤ $500: may be auto-approved by the agent if fully compliant.",
    "POL-APR-02": "Manager tier — Total > $500 and ≤ $2,000: eligible for approval, treated as approvable when fully compliant.",
    "POL-APR-03": "Director / Manual-Review tier — Total > $2,000: exceeds the agent's auto-approval authority and must be routed to Manual Review (director approval required), even if otherwise compliant.",
    "POL-TIME-01": "Submission window — Claims must be submitted within 30 days of the expense date. Late claims are routed to Manual Review.",
}
DECISION_GUIDANCE = "Approve: every item eligible, all receipts present, all within per-diem, total within an approvable tier. Partially Approve: claim valid but some amounts exceed caps; reimburse up to the cap. Reject: items ineligible with nothing reimbursable. Manual Review: any ambiguity, policy exception, high value, missing required receipt, or conflicting information. Prefer Manual Review over forcing a decision. Approval thresholds use reimbursable amounts after deductions."
LIMIT_TABLE = {"meals": {"amount": "75.00", "unit": "day", "ref": "POL-PD-01"}, "lodging": {"amount": "200.00", "unit": "night", "ref": "POL-PD-02"}, "ground_transport": {"amount": "50.00", "unit": "day", "ref": "POL-PD-03"}}
SAMPLE_CLAIMS = [
    {"claim_id": "CLM-001", "employee": "A. Rivera", "purpose": "Attend 2-day industry conference (business)", "business_documented": True, "trip_start": "2026-06-10", "trip_end": "2026-06-12", "submitted": "2026-06-20", "total_claimed": "1110.00", "currency": "USD", "items": [
        {"item_id": "I1", "category": "airfare", "description": "Round-trip economy airfare", "amount": "420.00", "receipt_attached": True, "airfare_class": "economy"},
        {"item_id": "I2", "category": "lodging", "description": "Hotel, 2 nights @ $180", "amount": "360.00", "receipt_attached": True, "units": 2},
        {"item_id": "I3", "category": "meals", "description": "Meals, 3 days @ ~$60/day", "amount": "180.00", "receipt_attached": True, "units": 3},
        {"item_id": "I4", "category": "conference_fees", "description": "Conference registration", "amount": "150.00", "receipt_attached": True}]},
    {"claim_id": "CLM-002", "employee": "B. Osei", "purpose": "Weekend hotel stay", "business_documented": False, "trip_start": "2026-06-14", "trip_end": "2026-06-15", "submitted": "2026-06-25", "total_claimed": "380.00", "currency": "USD", "items": [
        {"item_id": "I1", "category": "spa", "description": "Hotel spa package", "amount": "300.00", "receipt_attached": True},
        {"item_id": "I2", "category": "minibar", "description": "In-room minibar", "amount": "80.00", "receipt_attached": True}]},
    {"claim_id": "CLM-003", "employee": "C. Nakamura", "purpose": "Client site visit (business)", "business_documented": True, "trip_start": "2026-06-08", "trip_end": "2026-06-10", "submitted": "2026-06-22", "total_claimed": "940.00", "currency": "USD", "items": [
        {"item_id": "I1", "category": "airfare", "description": "Round-trip economy airfare", "amount": "300.00", "receipt_attached": True, "airfare_class": "economy"},
        {"item_id": "I2", "category": "lodging", "description": "Hotel, 2 nights @ $250", "amount": "500.00", "receipt_attached": True, "units": 2},
        {"item_id": "I3", "category": "meals", "description": "Meals, 2 days @ $70/day", "amount": "140.00", "receipt_attached": True, "units": 2}]},
    {"claim_id": "CLM-004", "employee": "D. Fischer", "purpose": "International vendor negotiation (business)", "business_documented": True, "trip_start": "2026-06-16", "trip_end": "2026-06-18", "submitted": "2026-06-28", "total_claimed": "3000.00", "currency": "USD", "items": [
        {"item_id": "I1", "category": "airfare", "description": "Business-class international airfare", "amount": "2400.00", "receipt_attached": True, "airfare_class": "business"},
        {"item_id": "I2", "category": "lodging", "description": "Hotel, 3 nights", "amount": "600.00", "receipt_attached": False, "units": 3}]},
    {"claim_id": "CLM-005", "employee": "E. Haddad", "purpose": "Client dinner / business development", "business_documented": True, "trip_start": "2026-06-11", "trip_end": "2026-06-11", "submitted": "2026-06-24", "total_claimed": "220.00", "currency": "USD", "items": [
        {"item_id": "I1", "category": "meals", "description": "Client dinner for 4 (business development)", "amount": "220.00", "receipt_attached": False, "units": 1}]},
]

## Typed inputs and output contract

Pydantic validates structural input and rejects unknown fields, non-boolean receipt flags, duplicate item IDs, unsupported currency, and invalid amounts. `Decimal` preserves cents during calculations. Semantic conflicts, such as inconsistent totals or dates, produce manual review rather than automatic approval.

`ClaimResult` has exactly nine fields. Its decision is one of `APPROVE`, `PARTIAL_APPROVE`, `REJECT`, and `MANUAL_REVIEW`. Stable reason codes appear inside `explanation`; additional findings, provisional amounts, and pending balances stay in separate audit evidence.


In [3]:
CENT = Decimal("0.01")


def money(value: Any) -> Decimal:
    if isinstance(value, bool):
        raise ValueError("Boolean is not a monetary amount")
    try:
        result = Decimal(str(value))
    except (InvalidOperation, TypeError, ValueError):
        raise ValueError("Invalid monetary amount") from None
    if not result.is_finite() or result < 0 or result > 10000000:
        raise ValueError("USD amount must be finite, non-negative and have at most two decimal places")
    try:
        rounded = result.quantize(CENT)
    except InvalidOperation:
        raise ValueError("Invalid monetary precision") from None
    if result != rounded:
        raise ValueError("USD amount must have at most two decimal places")
    return rounded


def jsonable(value: Any) -> Any:
    if isinstance(value, BaseModel):
        return jsonable(value.model_dump())
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    if isinstance(value, Enum):
        return value.value
    if isinstance(value, dict):
        return {k: jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    return value


def fingerprint(value: Any) -> str:
    return hashlib.sha256(json.dumps(jsonable(value), sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode()).hexdigest()


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid", validate_assignment=True)


class Expense(StrictModel):
    item_id: str = Field(min_length=1, max_length=40)
    category: str = Field(min_length=1, max_length=60)
    description: str = Field(min_length=1, max_length=1000)
    amount: Decimal = Field(gt=0)
    receipt_attached: bool = Field(strict=True)
    receipt_itemized: bool = Field(default=True, strict=True)  # Appendix B's Yes is assumed itemized, not OCR-verified.
    units: int | None = Field(default=None, ge=1, le=366, strict=True)
    expense_date: date | None = None
    airfare_class: Literal["economy", "business", "first"] | None = None
    _amount = field_validator("amount", mode="before")(money)


class Claim(StrictModel):
    claim_id: str = Field(min_length=1, max_length=60, pattern=r"^[A-Za-z0-9_-]+$")
    employee: str = Field(min_length=1, max_length=100)
    purpose: str = Field(min_length=1, max_length=2000)
    business_documented: bool = Field(strict=True)
    trip_start: date
    trip_end: date
    submitted: date
    currency: Literal["USD"] = "USD"
    total_claimed: Decimal = Field(gt=0)
    items: list[Expense] = Field(min_length=1, max_length=50)
    _total = field_validator("total_claimed", mode="before")(money)

    @model_validator(mode="after")
    def unique_items(self):
        if len({x.item_id for x in self.items}) != len(self.items):
            raise ValueError("Each item_id must be unique within its claim")
        return self


class Decision(str, Enum):
    APPROVE = "APPROVE"
    PARTIAL_APPROVE = "PARTIAL_APPROVE"
    REJECT = "REJECT"
    MANUAL_REVIEW = "MANUAL_REVIEW"


REASON_CODES = frozenset({
    "POLICY_COMPLIANT", "LIMIT_EXCEEDED", "INELIGIBLE_EXPENSE",
    "RECEIPT_MISSING", "AIRFARE_CLASS_EXCEPTION", "DIRECTOR_APPROVAL_REQUIRED",
    "LATE_SUBMISSION", "CONFLICTING_INFORMATION", "POLICY_AMBIGUITY",
    "MODEL_UNAVAILABLE", "OUTPUT_VALIDATION_FAILED",
})


class ClaimResult(StrictModel):
    claim_id: str
    decision: Decision
    approved_amount: Decimal
    deducted_amount: Decimal
    missing_docs: list[str]
    policy_refs: list[str]
    confidence: float = Field(ge=0, le=1)
    explanation: str = Field(min_length=1, max_length=12000)
    tools_used: list[str]
    _amounts = field_validator("approved_amount", "deducted_amount", mode="before")(money)


class Finding(StrictModel):
    code: str
    message: str
    policy_refs: list[str]
    item_ids: list[str] = Field(default_factory=list)
    review: bool = False


def finding(code: str, message: str, refs: list[str], items: list[str] | None = None, review: bool = False) -> dict:
    if code not in REASON_CODES or not set(refs).issubset(POLICIES):
        raise ValueError("Unknown reason code or policy citation")
    return Finding(code=code, message=message, policy_refs=refs, item_ids=items or [], review=review).model_dump()


class PolicyRepository:
    """Explicit lookup is sufficient for twelve fixed rules; no vector database."""
    def __init__(self):
        self.rules = dict(POLICIES)
        self.version = fingerprint({"rules": self.rules, "limits": LIMIT_TABLE, "guidance": DECISION_GUIDANCE})

    def lookup(self, refs: list[str] | None = None) -> dict:
        refs = list(self.rules) if refs is None else refs
        if any(ref not in self.rules for ref in refs):
            raise ValueError("Unknown policy reference")
        return {"policy_version": self.version, "rules": {ref: self.rules[ref] for ref in refs}, "decision_guidance": DECISION_GUIDANCE}

## Policy functions and decision control

`PolicyRepository` retrieves authoritative rule context before evaluation. `PolicyEvaluator` implements five focused functions for eligibility, receipts, limits, approval authority, and timeliness. `tools_used` lists the functions actually invoked by this evaluation; it is not a fabricated model tool trace.

Manual-review triggers take precedence. Only a fully resolved claim can be approved, partially approved, or rejected. Required receipts are checked per original line item before daily expenses are aggregated. Multiple same-day expenses share one allowance. Potentially overlapping undated amounts require clarification.


In [4]:
class PolicyEvaluator:
    ELIGIBLE = {"airfare", "lodging", "meals", "ground_transport", "conference_fees"}
    INELIGIBLE = {"alcohol", "minibar", "spa", "gym", "personal_entertainment", "in_room_movies", "personal_shopping", "gifts", "traffic_fines", "penalties", "late_fees", "personal"}

    def __init__(self, repository: PolicyRepository | None = None):
        self.repository = repository or PolicyRepository()

    def check_eligibility(self, claim: Claim) -> dict:
        findings = []
        for item in claim.items:
            if item.category in self.INELIGIBLE:
                findings.append(finding("INELIGIBLE_EXPENSE", f"{item.item_id}: {item.category} is deducted in full (${item.amount:.2f}).", ["POL-CAT-02"], [item.item_id]))
            elif item.category not in self.ELIGIBLE:
                findings.append(finding("POLICY_AMBIGUITY", f"{item.item_id}: category '{item.category}' is not resolved by the policy.", ["POL-CAT-01", "POL-CAT-02"], [item.item_id], True))
            else:
                if not claim.business_documented:
                    findings.append(finding("POLICY_AMBIGUITY", f"{item.item_id}: documented business purpose is missing.", ["POL-CAT-01"], [item.item_id], True))
                if item.category == "airfare":
                    if item.airfare_class in {"business", "first"}:
                        findings.append(finding("AIRFARE_CLASS_EXCEPTION", f"{item.item_id}: {item.airfare_class}-class airfare requires checking pre-approval; do not automatically deduct it.", ["POL-AIR-01"], [item.item_id], True))
                    elif item.airfare_class is None:
                        findings.append(finding("POLICY_AMBIGUITY", f"{item.item_id}: airfare class is unspecified.", ["POL-AIR-01"], [item.item_id], True))
                    elif re.search(r"\b(business.class|first.class)\b", item.description, re.I):
                        findings.append(finding("CONFLICTING_INFORMATION", f"{item.item_id}: description conflicts with economy-class metadata.", ["POL-AIR-01"], [item.item_id], True))
        refs = []
        if any(i.category in self.ELIGIBLE or i.category not in self.INELIGIBLE for i in claim.items):
            refs.append("POL-CAT-01")
        if any(i.category not in self.ELIGIBLE for i in claim.items):
            refs.append("POL-CAT-02")
        if any(i.category == "airfare" for i in claim.items):
            refs.append("POL-AIR-01")
        return {"findings": findings, "policy_refs": refs}

    def check_receipts(self, claim: Claim) -> dict:
        findings, missing = [], []
        for item in claim.items:
            if item.amount > 25 or item.category in {"airfare", "lodging"}:
                if not item.receipt_attached or not item.receipt_itemized:
                    doc = f"{item.item_id}: itemized {item.category} receipt"
                    missing.append(doc)
                    findings.append(finding("RECEIPT_MISSING", f"{doc} is required and {'missing' if not item.receipt_attached else 'not itemized'}; request it before deciding.", ["POL-RCT-01", "POL-RCT-02"], [item.item_id], True))
        return {"missing_docs": missing, "findings": findings, "policy_refs": ["POL-RCT-01"] + (["POL-RCT-02"] if missing else [])}

    def calculate_limits(self, claim: Claim) -> dict:
        findings, lines, groups = [], [], defaultdict(list)
        trip_days = (claim.trip_end - claim.trip_start).days + 1
        for item in claim.items:
            eligible = item.category in self.ELIGIBLE
            limit = LIMIT_TABLE.get(item.category)
            permitted = item.amount if eligible or item.category not in self.INELIGIBLE else Decimal(0)
            line = {"item_id": item.item_id, "category": item.category, "claimed": item.amount, "provisional_allowed": permitted, "provisional_deducted": item.amount - permitted, "policy_refs": ["POL-CAT-02"] if item.category in self.INELIGIBLE else []}
            lines.append(line)
            if limit:
                units = item.units or (1 if item.expense_date else None)
                if units is None:
                    findings.append(finding("POLICY_AMBIGUITY", f"{item.item_id}: number of {limit['unit']}s is missing; no cap is invented.", [limit["ref"]], [item.item_id], True))
                    continue
                maximum_units = max(0, trip_days - 1) if item.category == "lodging" else trip_days
                if units > maximum_units:
                    findings.append(finding("CONFLICTING_INFORMATION", f"{item.item_id}: stated {units} {limit['unit']}(s) exceed the trip's {maximum_units}; retain the supplied count and verify dates.", [limit["ref"]], [item.item_id], True))
                if item.expense_date and units > 1:
                    findings.append(finding("POLICY_AMBIGUITY", f"{item.item_id}: a multi-day amount needs a dated breakdown.", [limit["ref"]], [item.item_id], True))
                groups[(item.category, item.expense_date)].append((item, line, units))
        for (category, expense_date), grouped in groups.items():
            limit = LIMIT_TABLE[category]
            # Separate undated aggregate rows may overlap. Do not multiply caps per row.
            if expense_date is None and len(grouped) > 1:
                findings.append(finding("POLICY_AMBIGUITY", f"Multiple undated {category} rows may overlap; request daily/nightly allocation.", [limit["ref"]], [x[0].item_id for x in grouped], True))
                continue
            units = grouped[0][2] if expense_date is None else 1
            remaining = Decimal(limit["amount"]) * units
            for item, line, _ in grouped:
                allowed = min(item.amount, max(Decimal(0), remaining))
                remaining -= allowed
                excess = item.amount - allowed
                line.update(provisional_allowed=allowed, provisional_deducted=excess, policy_refs=[limit["ref"]])
                if excess:
                    findings.append(finding("LIMIT_EXCEEDED", f"{item.item_id}: ${item.amount:.2f} claimed; ${allowed:.2f} allowed after the ${limit['amount']} per-{limit['unit']} cap; ${excess:.2f} excess.", [limit["ref"]], [item.item_id]))
        for category in LIMIT_TABLE:
            category_items = [i for i in claim.items if i.category == category]
            if any(i.expense_date is None for i in category_items) and any(i.expense_date is not None for i in category_items):
                findings.append(finding("POLICY_AMBIGUITY", f"Dated and undated {category} rows may overlap; obtain a daily/nightly breakdown before applying caps.", [LIMIT_TABLE[category]["ref"]], [i.item_id for i in category_items], True))
        return jsonable({"lines": lines, "provisional_allowed": sum(x["provisional_allowed"] for x in lines), "provisional_deducted": sum(x["provisional_deducted"] for x in lines), "findings": findings, "policy_refs": sorted({r for x in lines for r in x["policy_refs"]})})

    def check_timeliness(self, claim: Claim) -> dict:
        findings = []
        if claim.trip_end < claim.trip_start:
            findings.append(finding("CONFLICTING_INFORMATION", "Trip end precedes trip start.", ["POL-TIME-01"], review=True))
        if sum(x.amount for x in claim.items) != claim.total_claimed:
            findings.append(finding("CONFLICTING_INFORMATION", "Declared total does not equal the sum of line items; reconcile the claim.", [], review=True))
        for item in claim.items:
            start = item.expense_date or claim.trip_start
            end = item.expense_date or claim.trip_end
            if claim.submitted < end:
                findings.append(finding("CONFLICTING_INFORMATION", f"{item.item_id}: submission predates the stated expense interval.", ["POL-TIME-01"], [item.item_id], True))
            elif (claim.submitted - end).days > 30:
                findings.append(finding("LATE_SUBMISSION", f"{item.item_id}: even the latest possible expense date is more than 30 days before submission.", ["POL-TIME-01"], [item.item_id], True))
            elif (claim.submitted - start).days > 30:
                findings.append(finding("POLICY_AMBIGUITY", f"{item.item_id}: trip interval crosses the 30-day deadline; obtain exact expense date.", ["POL-TIME-01"], [item.item_id], True))
        return {"findings": findings, "policy_refs": ["POL-TIME-01"]}

    def check_approval_threshold(self, claim: Claim) -> dict:
        limits = self.calculate_limits(claim)
        eligibility = self.check_eligibility(claim)
        unresolved = any(f["review"] for f in eligibility["findings"] + limits["findings"])
        amount = money(limits["provisional_allowed"])
        if unresolved:
            return {"tier": "UNRESOLVED", "candidate_amount": float(amount), "findings": [], "policy_refs": [], "note": "Provisional amount depends on unresolved eligibility or limits; do not infer director authority from gross claim amount."}
        ref = "POL-APR-01" if amount <= 500 else "POL-APR-02" if amount <= 2000 else "POL-APR-03"
        findings = [finding("DIRECTOR_APPROVAL_REQUIRED", f"Reimbursable amount ${amount:.2f} exceeds $2,000; director approval required.", [ref], review=True)] if amount > 2000 else []
        return {"tier": {"POL-APR-01": "AUTO", "POL-APR-02": "MANAGER", "POL-APR-03": "DIRECTOR"}[ref], "candidate_amount": float(amount), "findings": findings, "policy_refs": [ref]}

    def evaluate(self, claim: Claim) -> dict:
        policy_context = self.repository.lookup()
        evidence = {name: getattr(self, name)(claim) for name in CHECK_NAMES}
        findings = [f for block in evidence.values() for f in block["findings"]]
        refs = sorted({r for block in evidence.values() for r in block["policy_refs"]} | {r for f in findings for r in f["policy_refs"]})
        limits = evidence["calculate_limits"]
        review = any(f["review"] for f in findings)
        allowed, deducted = money(limits["provisional_allowed"]), money(limits["provisional_deducted"])
        decision = Decision.MANUAL_REVIEW if review else Decision.REJECT if allowed == 0 else Decision.PARTIAL_APPROVE if deducted else Decision.APPROVE
        if decision == Decision.APPROVE:
            findings.append(finding("POLICY_COMPLIANT", f"All items are eligible, documented, timely and within limits; ${allowed:.2f} is approvable in the {evidence['check_approval_threshold']['tier'].lower()} tier.", refs))
        sentences = [f"{f['code']}: {f['message']}" + (f" ({', '.join(f['policy_refs'])})" if f["policy_refs"] else "") for f in findings]
        if review:
            sentences.append(f"The full ${claim.total_claimed:.2f} remains pending; approved and deducted amounts are both zero until review. Calculated allowances are provisional.")
        elif decision != Decision.APPROVE:
            sentences.append(f"Final recommendation: {decision.value}; ${allowed:.2f} approved and ${deducted:.2f} deducted. Supporting checks: {', '.join(refs)}.")
        ambiguous = any(f["code"] in {"POLICY_AMBIGUITY", "CONFLICTING_INFORMATION"} for f in findings)
        result = ClaimResult(claim_id=claim.claim_id, decision=decision, approved_amount=0 if review else allowed, deducted_amount=0 if review else deducted, missing_docs=evidence["check_receipts"]["missing_docs"], policy_refs=refs, confidence=0.70 if ambiguous else 0.95, explanation=" ".join(sentences), tools_used=list(TOOL_NAMES))
        return {"result": result, "policy_context": policy_context, "findings": findings, "checks": evidence, "lines": limits["lines"], "pending_amount": float(claim.total_claimed) if review else 0, "reason_codes": sorted({f["code"] for f in findings})}


CHECK_NAMES = ("check_eligibility", "check_receipts", "calculate_limits", "check_approval_threshold", "check_timeliness")
TOOL_NAMES = ("lookup_policy",) + CHECK_NAMES

## Provided-input verification

Validate the source transcription and declared totals before evaluating outcomes.


In [5]:
CLAIMS = [Claim.model_validate(raw) for raw in SAMPLE_CLAIMS]
EXPECTED_TOTALS = {"CLM-001": "1110.00", "CLM-002": "380.00", "CLM-003": "940.00", "CLM-004": "3000.00", "CLM-005": "220.00"}
assert len(POLICIES) == 12 and len(CLAIMS) == 5
assert [c.claim_id for c in CLAIMS] == list(EXPECTED_TOTALS)
for claim in CLAIMS:
    assert claim.total_claimed == Decimal(EXPECTED_TOTALS[claim.claim_id])
    assert sum(item.amount for item in claim.items) == claim.total_claimed
assert sum(c.total_claimed for c in CLAIMS) == Decimal("5650.00")
print("Verified: 12 policies, 5 supplied claims, and $5,650.00 in reconciled claimed amounts.")


Verified: 12 policies, 5 supplied claims, and $5,650.00 in reconciled claimed amounts.


## Generated decisions and supporting evidence

The following results are generated by the policy evaluator. **Manual review holds the entire claim**: approval and final deduction are both zero, while possible allowances and deductions remain provisional. The pending amount is display/audit metadata, not an extra result field.

Expand a claim to inspect its explanation, cited rules, check findings, and line calculations. The allowance calculation for CLM-003 is **$300 airfare + $400 lodging + $140 meals = $840**. For CLM-004, a business-class exception prevents a definitive approval-tier calculation; gross $3,000 is not treated as established reimbursable value.


In [6]:
from IPython.display import HTML, display

EVALUATOR = PolicyEvaluator()
EVALUATIONS = [EVALUATOR.evaluate(claim) for claim in CLAIMS]
RESULTS = [jsonable(e["result"]) for e in EVALUATIONS]
claimed = sum(c.total_claimed for c in CLAIMS)
approved = sum(e["result"].approved_amount for e in EVALUATIONS)
deducted = sum(e["result"].deducted_amount for e in EVALUATIONS)
pending = sum(money(e["pending_amount"]) for e in EVALUATIONS)
assert claimed == approved + deducted + pending

rows, details = [], []
for claim, evidence in zip(CLAIMS, EVALUATIONS):
    result = evidence["result"]
    rows.append(f"<tr><td>{html.escape(claim.claim_id)}</td><td>{html.escape(claim.employee)}</td><td>{result.decision.value}</td><td>${result.approved_amount:,.2f}</td><td>${result.deducted_amount:,.2f}</td><td>${evidence['pending_amount']:,.2f}</td></tr>")
    citations = "".join(f"<li><strong>{ref}</strong>: {html.escape(POLICIES[ref])}</li>" for ref in result.policy_refs)
    checks = html.escape(json.dumps(jsonable({"checks": evidence["checks"], "provisional_lines": evidence["lines"]}), indent=2))
    details.append(f"<details><summary><strong>{claim.claim_id}</strong> — {result.decision.value}</summary><p>{html.escape(result.explanation)}</p><p><strong>Missing documents:</strong> {html.escape(', '.join(result.missing_docs) or 'None')}</p><p><strong>Reason codes:</strong> {html.escape(', '.join(evidence['reason_codes']))}</p><ul>{citations}</ul><details><summary>Check findings and provisional calculations</summary><pre>{checks}</pre></details></details>")
display(HTML(f"""<div style="font-family:system-ui;line-height:1.6;color:#17233c;background:#fff;padding:20px;border:1px solid #dce2eb;border-radius:12px">
<h3 style="margin-top:0">Reimbursement outcomes</h3>
<p>Python policy evaluation · USD · Five supplied claims</p>
<p><strong>Claimed ${claimed:,.2f}</strong> &nbsp; Approved ${approved:,.2f} &nbsp; Deducted ${deducted:,.2f} &nbsp; Pending ${pending:,.2f}</p>
<div style="overflow-x:auto"><table style="border-collapse:collapse;width:100%;text-align:left"><thead><tr><th>Claim</th><th>Employee</th><th>Decision</th><th>Approved</th><th>Deducted</th><th>Pending</th></tr></thead><tbody>{''.join(rows)}</tbody></table></div>
<hr>{''.join(details)}</div>"""))


Claim,Employee,Decision,Approved,Deducted,Pending
CLM-001,A. Rivera,APPROVE,"$1,110.00",$0.00,$0.00
CLM-002,B. Osei,REJECT,$0.00,$380.00,$0.00
CLM-003,C. Nakamura,PARTIAL_APPROVE,$840.00,$100.00,$0.00
CLM-004,D. Fischer,MANUAL_REVIEW,$0.00,$0.00,"$3,000.00"
CLM-005,E. Haddad,MANUAL_REVIEW,$0.00,$0.00,$220.00


## Automated policy checks

Run acceptance checks for all five claims and boundary checks for receipts, caps, approval tiers, timeliness, aggregation, malformed input, citations, and review precedence. Controlled variations test individual rules; they do not modify or replace the five demonstration claims.

The test code is embedded so the downloaded notebook can reproduce its validation. Tests verify independently stated outcomes and business invariants rather than comparing the evaluator with itself.


In [7]:
import io
import unittest


class PolicyTests(unittest.TestCase):
    """Controlled variants test rules; SAMPLE_CLAIMS remains the only demo dataset."""
    def setUp(self):
        self.evaluator = PolicyEvaluator()

    def single(self, category="conference_fees", amount="75.00", receipt=True, **item_fields):
        raw = copy.deepcopy(SAMPLE_CLAIMS[0])
        item = {"item_id": "I1", "category": category, "description": "Boundary variation of the supplied claim", "amount": amount, "receipt_attached": receipt}
        item.update(item_fields)
        raw.update(items=[item], total_claimed=amount)
        return raw

    def evaluate(self, raw):
        return self.evaluator.evaluate(Claim.model_validate(raw))

    def assert_outcome(self, raw, decision, approved, deducted):
        evidence = self.evaluate(raw)
        result = evidence["result"]
        self.assertEqual(result.decision.value, decision)
        self.assertEqual(result.approved_amount, Decimal(str(approved)))
        self.assertEqual(result.deducted_amount, Decimal(str(deducted)))
        self.assertTrue(set(result.policy_refs).issubset(POLICIES))
        self.assertTrue(set(evidence["reason_codes"]).issubset(REASON_CODES))
        for code in evidence["reason_codes"]:
            self.assertIn(code + ":", result.explanation)
        for f in evidence["findings"]:
            for ref in f["policy_refs"]:
                self.assertIn(ref, result.policy_refs)
                self.assertIn(ref, result.explanation)
        self.assertEqual(result.tools_used, list(TOOL_NAMES))
        self.assertEqual(result.approved_amount + result.deducted_amount + money(evidence["pending_amount"]), money(raw["total_claimed"]))
        return evidence

    def test_all_five_supplied_results_and_exact_schema(self):
        expected = [("APPROVE", "1110", "0"), ("REJECT", "0", "380"), ("PARTIAL_APPROVE", "840", "100"), ("MANUAL_REVIEW", "0", "0"), ("MANUAL_REVIEW", "0", "0")]
        required = {"claim_id", "decision", "approved_amount", "deducted_amount", "missing_docs", "policy_refs", "confidence", "explanation", "tools_used"}
        for raw, outcome in zip(SAMPLE_CLAIMS, expected):
            with self.subTest(claim=raw["claim_id"]):
                evidence = self.assert_outcome(raw, *outcome)
                self.assertEqual(set(jsonable(evidence["result"])), required)
                self.assertEqual(jsonable(evidence["result"])["claim_id"], raw["claim_id"])

    def test_sample_monetary_reconciliation(self):
        evaluations = [self.evaluate(c) for c in SAMPLE_CLAIMS]
        self.assertEqual(sum(e["result"].approved_amount for e in evaluations), Decimal("1950.00"))
        self.assertEqual(sum(e["result"].deducted_amount for e in evaluations), Decimal("480.00"))
        self.assertEqual(sum(money(e["pending_amount"]) for e in evaluations), Decimal("3220.00"))

    def test_missing_receipt_names_and_manual_review_precedence(self):
        four, five = [self.evaluate(SAMPLE_CLAIMS[i]) for i in (3, 4)]
        self.assertEqual(four["result"].missing_docs, ["I2: itemized lodging receipt"])
        self.assertEqual(five["result"].missing_docs, ["I1: itemized meals receipt"])
        self.assertEqual(money(five["lines"][0]["provisional_deducted"]), Decimal("145"))
        self.assertEqual(five["result"].deducted_amount, 0)
        self.assertNotIn("POL-APR-03", four["result"].policy_refs)
        self.assertEqual(four["checks"]["check_approval_threshold"]["tier"], "UNRESOLVED")

    def test_receipt_threshold_is_strictly_above_25(self):
        self.assert_outcome(self.single(amount="25.00", receipt=False), "APPROVE", "25", "0")
        e = self.assert_outcome(self.single(amount="25.01", receipt=False), "MANUAL_REVIEW", "0", "0")
        self.assertIn("RECEIPT_MISSING", e["reason_codes"])

    def test_airfare_and_lodging_always_require_receipts(self):
        for category, fields in [("airfare", {"airfare_class": "economy"}), ("lodging", {"units": 1})]:
            with self.subTest(category=category):
                self.assert_outcome(self.single(category, "20", False, **fields), "MANUAL_REVIEW", "0", "0")

    def test_attached_non_itemized_receipt_requires_review(self):
        e = self.assert_outcome(self.single(receipt_itemized=False), "MANUAL_REVIEW", "0", "0")
        self.assertIn("not itemized", e["result"].explanation)

    def test_approval_boundaries(self):
        cases = [("500", "APPROVE", "AUTO"), ("500.01", "APPROVE", "MANAGER"), ("2000", "APPROVE", "MANAGER"), ("2000.01", "MANUAL_REVIEW", "DIRECTOR")]
        for amount, decision, tier in cases:
            with self.subTest(amount=amount):
                e = self.assert_outcome(self.single(amount=amount), decision, amount if decision == "APPROVE" else "0", "0")
                self.assertEqual(e["checks"]["check_approval_threshold"]["tier"], tier)

    def test_approval_threshold_uses_post_cap_amount(self):
        e = self.assert_outcome(self.single("lodging", "2500", units=2), "PARTIAL_APPROVE", "400", "2100")
        self.assertEqual(e["checks"]["check_approval_threshold"]["tier"], "AUTO")
        self.assertNotIn("DIRECTOR_APPROVAL_REQUIRED", e["reason_codes"])

    def test_category_caps_and_cent_boundaries(self):
        for category, cap in [("meals", Decimal("75")), ("lodging", Decimal("200")), ("ground_transport", Decimal("50"))]:
            for excess in (Decimal("0"), Decimal("0.01")):
                with self.subTest(category=category, excess=excess):
                    raw = self.single(category, str(cap + excess), units=1)
                    self.assert_outcome(raw, "APPROVE" if not excess else "PARTIAL_APPROVE", cap, excess)

    def test_category_specific_counts(self):
        e = self.evaluate(SAMPLE_CLAIMS[2])
        self.assertEqual([money(x["provisional_allowed"]) for x in e["lines"]], [Decimal("300"), Decimal("400"), Decimal("140")])

    def test_all_ineligible_categories_are_deducted(self):
        for category in PolicyEvaluator.INELIGIBLE:
            with self.subTest(category=category):
                self.assert_outcome(self.single(category, "10", False), "REJECT", "0", "10")

    def test_mixed_eligible_and_ineligible_items(self):
        raw = self.single(amount="100")
        raw["items"].append({"item_id": "I2", "category": "alcohol", "description": "Boundary variation", "amount": "10", "receipt_attached": False})
        raw["total_claimed"] = "110"
        self.assert_outcome(raw, "PARTIAL_APPROVE", "100", "10")

    def test_unknown_category_requires_review(self):
        e = self.assert_outcome(self.single("unrecognized_category"), "MANUAL_REVIEW", "0", "0")
        self.assertIn("POLICY_AMBIGUITY", e["reason_codes"])

    def test_missing_business_purpose_requires_review(self):
        raw = self.single()
        raw["business_documented"] = False
        self.assert_outcome(raw, "MANUAL_REVIEW", "0", "0")

    def test_airfare_classes_and_conflicting_description(self):
        for airfare_class in ("business", "first", None):
            with self.subTest(airfare_class=airfare_class):
                self.assert_outcome(self.single("airfare", "300", airfare_class=airfare_class), "MANUAL_REVIEW", "0", "0")
        raw = self.single("airfare", "300", airfare_class="economy", description="Business-class international airfare")
        e = self.assert_outcome(raw, "MANUAL_REVIEW", "0", "0")
        self.assertIn("CONFLICTING_INFORMATION", e["reason_codes"])

    def test_timeliness_30_and_31_days(self):
        for elapsed, decision in [(30, "APPROVE"), (31, "MANUAL_REVIEW")]:
            with self.subTest(elapsed=elapsed):
                raw = self.single(expense_date="2026-06-10")
                raw["submitted"] = "2026-07-10" if elapsed == 30 else "2026-07-11"
                self.assert_outcome(raw, decision, "75" if elapsed == 30 else "0", "0")

    def test_trip_interval_crossing_timeliness_boundary(self):
        raw = self.single()
        raw["submitted"] = "2026-07-11"
        e = self.assert_outcome(raw, "MANUAL_REVIEW", "0", "0")
        self.assertIn("POLICY_AMBIGUITY", e["reason_codes"])

    def test_all_possible_dates_late(self):
        raw = self.single()
        raw["submitted"] = "2026-07-13"
        e = self.assert_outcome(raw, "MANUAL_REVIEW", "0", "0")
        self.assertIn("LATE_SUBMISSION", e["reason_codes"])

    def test_submission_before_expense_and_reversed_trip(self):
        for changes in [{"submitted": "2026-06-09"}, {"trip_end": "2026-06-09"}]:
            with self.subTest(changes=changes):
                raw = self.single()
                raw.update(changes)
                e = self.assert_outcome(raw, "MANUAL_REVIEW", "0", "0")
                self.assertIn("CONFLICTING_INFORMATION", e["reason_codes"])

    def test_total_mismatch_requires_review(self):
        raw = self.single()
        raw["total_claimed"] = "76"
        e = self.assert_outcome(raw, "MANUAL_REVIEW", "0", "0")
        self.assertIn("CONFLICTING_INFORMATION", e["reason_codes"])

    def test_missing_or_conflicting_quantity(self):
        for fields in [{}, {"units": 4}, {"units": 2, "expense_date": "2026-06-10"}]:
            with self.subTest(fields=fields):
                self.assert_outcome(self.single("meals", "140", **fields), "MANUAL_REVIEW", "0", "0")

    def test_multiple_same_day_items_share_one_cap(self):
        raw = self.single("meals", "40", expense_date="2026-06-10")
        raw["items"].append({"item_id": "I2", "category": "meals", "description": "Boundary variation", "amount": "50", "receipt_attached": True, "expense_date": "2026-06-10"})
        raw["total_claimed"] = "90"
        self.assert_outcome(raw, "PARTIAL_APPROVE", "75", "15")

    def test_distinct_dated_days_receive_distinct_caps(self):
        raw = self.single("meals", "70", expense_date="2026-06-10")
        raw["items"].append({"item_id": "I2", "category": "meals", "description": "Boundary variation", "amount": "70", "receipt_attached": True, "expense_date": "2026-06-11"})
        raw["total_claimed"] = "140"
        self.assert_outcome(raw, "APPROVE", "140", "0")

    def test_overlapping_undated_rows_require_review(self):
        for dated_second in (False, True):
            with self.subTest(dated_second=dated_second):
                raw = self.single("meals", "70", units=1)
                item = {"item_id": "I2", "category": "meals", "description": "Boundary variation", "amount": "70", "receipt_attached": True, "units": 1}
                if dated_second:
                    item["expense_date"] = "2026-06-10"
                raw["items"].append(item)
                raw["total_claimed"] = "140"
                self.assert_outcome(raw, "MANUAL_REVIEW", "0", "0")

    def test_invalid_money_is_rejected(self):
        values = [True, None, "NaN", "Infinity", "-1", "0.001", "1e1000", "not-money", [], {}]
        for value in values:
            with self.subTest(value=value):
                with self.assertRaises(ValueError):
                    money(value)
        self.assertEqual(money("0.10") + money("0.20"), Decimal("0.30"))

    def test_zero_amount_and_non_boolean_flags_rejected(self):
        for raw in [self.single(amount="0"), self.single(receipt="yes"), self.single(units=True)]:
            with self.subTest(raw=raw):
                with self.assertRaises(ValueError):
                    Claim.model_validate(raw)
        raw = self.single()
        raw["business_documented"] = "true"
        with self.assertRaises(ValueError):
            Claim.model_validate(raw)

    def test_extra_fields_currency_and_duplicate_items_rejected(self):
        for changes in [{"currency": "INR"}, {"unexpected_field": True}]:
            raw = self.single()
            raw.update(changes)
            with self.assertRaises(ValueError):
                Claim.model_validate(raw)
        raw = self.single()
        raw["items"].append(copy.deepcopy(raw["items"][0]))
        with self.assertRaises(ValueError):
            Claim.model_validate(raw)

    def test_unknown_citations_and_reason_codes_rejected(self):
        with self.assertRaises(ValueError):
            PolicyRepository().lookup(["POL-FAKE-01"])
        with self.assertRaises(ValueError):
            finding("FAKE_REASON", "bad", [])
        with self.assertRaises(ValueError):
            finding("POLICY_AMBIGUITY", "bad", ["POL-FAKE-01"])
        self.assertEqual(PolicyRepository().lookup([])["rules"], {})

    def test_confidence_is_evidence_strength_not_approval_probability(self):
        clear_review = self.evaluate(SAMPLE_CLAIMS[4])["result"]
        ambiguous_review = self.evaluate(SAMPLE_CLAIMS[3])["result"]
        self.assertEqual(clear_review.confidence, 0.95)
        self.assertEqual(ambiguous_review.confidence, 0.70)
        self.assertEqual(clear_review.decision, Decision.MANUAL_REVIEW)

    def test_input_and_policy_are_not_modified(self):
        before_claims, before_rules = fingerprint(SAMPLE_CLAIMS), fingerprint(POLICIES)
        for raw in SAMPLE_CLAIMS:
            self.evaluate(raw)
        self.assertEqual(fingerprint(SAMPLE_CLAIMS), before_claims)
        self.assertEqual(fingerprint(POLICIES), before_rules)
        self.assertEqual(PolicyRepository().version, PolicyRepository().version)

    def test_json_serialization_and_schema_validation(self):
        result = self.evaluate(SAMPLE_CLAIMS[2])["result"]
        exported = jsonable(result)
        self.assertEqual(ClaimResult.model_validate(exported), result)
        self.assertEqual(json.loads(json.dumps(exported))["approved_amount"], 840.0)
        self.assertEqual(jsonable((date(2026, 6, 10), Decision.APPROVE)), ["2026-06-10", "APPROVE"])


def run_policy_tests() -> dict:
    stream = io.StringIO()
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(PolicyTests)
    report = unittest.TextTestRunner(stream=stream, verbosity=1).run(suite)
    print(stream.getvalue().strip())
    if not report.wasSuccessful():
        raise AssertionError("Policy acceptance checks failed")
    return {"test_methods": report.testsRun, "failures": len(report.failures), "errors": len(report.errors), "status": "PASS"}

In [8]:
POLICY_TEST_REPORT = run_policy_tests()
print(json.dumps(POLICY_TEST_REPORT, indent=2))


...............................
----------------------------------------------------------------------
Ran 31 tests in 0.008s

OK
{
  "test_methods": 31,
  "failures": 0,
  "errors": 0,
  "status": "PASS"
}


## Design Notes & Reasoning

### Assumptions and decision semantics

- **Grounding:** use only the supplied Appendix A policy and Appendix B claims. Twelve fixed rules are small enough for explicit lookup; a vector database would add no useful retrieval capability here.
- **Business context:** structured business-purpose flags are transcribed from the supplied business scenarios. Undocumented eligible expenses require clarification; explicit ineligible categories are deducted in full.
- **Receipts:** a supplied “Yes” is assumed to mean an attached itemized receipt. No OCR or receipt authenticity verification is claimed. Exactly $25 does not require a receipt, except airfare and lodging, which always do.
- **Expense dates:** when no exact date is supplied, assume expenses occurred during the given trip interval. If that interval straddles the 30-day deadline, request the exact date. Do not compare submission dates against today's date.
- **Aggregate rows:** honor the supplied category-specific day/night counts and assume their stated aggregate rates describe the covered periods. Separate undated rows or a mix of dated and undated rows may overlap and require a daily/nightly breakdown.
- **CLM-004:** business-class airfare needs exception/pre-approval review (`POL-AIR-01`); the lodging receipt is missing (`POL-RCT-01`, `POL-RCT-02`); three stated hotel nights conflict with the trip interval (`POL-PD-02`). Preserve the source facts rather than silently reducing the night count. Director authority is not inferred from gross claimed value while reimbursement eligibility remains unresolved.
- **CLM-005:** the required dinner receipt is missing (`POL-RCT-02`). The $75 daily allowance is not multiplied by four attendees because the policy gives no such exception. $75 allowed/$145 excess is a provisional calculation; the entire $220 remains pending.
- **Manual review:** the full claim is held, with $0 approved and $0 finally deducted. Provisional line calculations stay in audit evidence. No human approval, payment, or document request is automatically executed.
- **Mixed claims:** if all information is resolved, deduct ineligible items and partially approve the eligible balance. Any review trigger takes precedence.
- **Confidence:** an evidence-strength indicator, not a calibrated probability of approval: `0.95` for a conclusive recommendation, including clear missing-receipt review; `0.70` for unresolved/conflicting facts. A model-execution failure will use `0.0` in the model-connected workflow.

### Implementation trade-offs and limitations

- Use typed models, ordinary Python classes, explicit checks, and `Decimal` rather than adding a general rules framework.
- Structural errors are rejected at intake. Semantic uncertainty yields `MANUAL_REVIEW` with stable reason codes and applicable policy citations.
- Monetary calculations and decision precedence are deterministic; the current execution does not claim live GenAI reasoning. This policy layer is designed to validate recommendations from the separate LangChain/MCP tool workflow.
- The policy is a fixed mock document. There is no currency conversion, durable claim history, real receipt verification, identity validation, or real approval system.
- Daily line allocation is stable in input order, while the total category/day allowance is shared. A production policy would need an explicit allocation convention and dated receipt details.
- Improvements would include verified itemized receipts, exact expense-date allocation, versioned policy changes, and evaluation across a broader authorized dataset. Study material and the later companion app stay outside the assessed artifact.

### References

1. Supplied *AI Developer Candidate Assignment — Travel Reimbursement Approval Agent*: functional expectations and deliverables, pp. 1–2; Appendix A policy, p. 3; Appendix B claims, p. 4. `POL-*` citations above refer to Appendix A.
2. [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/).
3. [Python decimal arithmetic](https://docs.python.org/3/library/decimal.html).
4. [LangChain ChatOllama integration](https://docs.langchain.com/oss/python/integrations/chat/ollama).
5. [Official MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk).

## Structured results

The final code cell prints only the JSON array of the five evaluated claims. Each object has exactly the required nine fields; pending amounts and structured audit details remain separate.


In [9]:
assert len(RESULTS) == 5
assert [r["claim_id"] for r in RESULTS] == [c.claim_id for c in CLAIMS]
assert all(set(r) == set(ClaimResult.model_fields) for r in RESULTS)
print(json.dumps(RESULTS, indent=2))


[
  {
    "claim_id": "CLM-001",
    "decision": "APPROVE",
    "approved_amount": 1110.0,
    "deducted_amount": 0.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-AIR-01",
      "POL-APR-02",
      "POL-CAT-01",
      "POL-PD-01",
      "POL-PD-02",
      "POL-RCT-01",
      "POL-TIME-01"
    ],
    "confidence": 0.95,
    "explanation": "POLICY_COMPLIANT: All items are eligible, documented, timely and within limits; $1110.00 is approvable in the manager tier. (POL-AIR-01, POL-APR-02, POL-CAT-01, POL-PD-01, POL-PD-02, POL-RCT-01, POL-TIME-01)",
    "tools_used": [
      "lookup_policy",
      "check_eligibility",
      "check_receipts",
      "calculate_limits",
      "check_approval_threshold",
      "check_timeliness"
    ]
  },
  {
    "claim_id": "CLM-002",
    "decision": "REJECT",
    "approved_amount": 0.0,
    "deducted_amount": 380.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-APR-01",
      "POL-CAT-02",
      "POL-RCT-01",
      "POL-TIME-01"
    ],
